In [ ]:
# !pip install -q -U google-generativeai

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
import os
os.chdir("/content/gdrive/MyDrive/LLM/CBLLMMODEL")

In [ ]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
for m in genai.list_models():
    print(m.name)

In [ ]:
model = genai.GenerativeModel('models/gemini-2.0-flash')

In [ ]:
def classify_sentence(sentence):
    prompt = f"""
You are an expert in monetary policy communication. Classify the following sentence from the Central Bank of the Republic of Turkey (CBRT) based on its policy tone.

Sentence: "{sentence}"

Choose only one of the following labels:

- "hawkish": if the sentence signals inflationary pressure, interest rate hikes, tightening, or economic risks requiring policy action.
- "dovish": if the sentence signals easing, financial support, rate cuts, or declining inflation.
- "neutral": if the sentence is purely descriptive with no policy implications.

   Return only the label as a single word: hawkish, dovish, or neutral.
"""

    try:
        response = model.generate_content(prompt)
        label = response.text.strip().lower()
        for valid_label in ["hawkish", "dovish", "neutral"]:
            if valid_label in label:
                return valid_label
        return "unknown"
    except Exception as e:
        print(f"Hata: {e}")
        return "hata"

In [ ]:
sentence = "tight"
classify_sentence(sentence)

In [ ]:
# TXT dosyasından cümleleri oku
import pandas as pd
with open("TCMB_SENTENCES.txt", "r", encoding="utf-8") as file:
    sentences = [line.strip() for line in file if line.strip()]

# DataFrame oluştur
df = pd.DataFrame(sentences, columns=["sentence"])
df["label"] = ""

In [ ]:
df.shape

In [ ]:
# Etiketleme döngüsü
import time
for idx, row in df.iterrows():
    label = classify_sentence(row["sentence"])
    df.at[idx, "label"] = label
    time.sleep(0.5)

    if idx % 100 == 0:
        df.to_excel("TCMB_LABELLED_GEMINI_PARTIAL.xlsx", index=False)
        print(f"{idx} cümle işlendi...")

    # print(idx)
# Son kaydetme
df.to_excel("TCMB_LABELLED_GEMINI_FINAL.xlsx", index=False)

In [ ]:
classify_sentence("Central bank decided to tightining money supply")